In [ ]:
# Reproducibility setup
import os, sys, random
import numpy as np
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
np.random.seed(42)
random.seed(42)


# 06 - LGD Final Layer

This notebook builds the **simplified final LGD layer** for downstream Expected Loss use. It standardises all four product datasets (Residential Mortgage, Commercial Cash Flow, Development Finance, Cash Flow Lending) into one facility-level output with:

- **Base LGD** by product and security type
- **LVR adjustment** for higher leverage exposures
- **Development stage adjustment** for earlier-stage projects
- **Industry risk adjustment** for higher-risk sectors
- **PD score band adjustment** for cash flow lending (A-E bands)
- **DSCR stress adjustment** for weak debt service coverage
- **Conduct overlay** for borrower conduct risk
- **Downturn scalar** to convert adjusted LGD into stressed LGD

The final output is a single `lgd_final` per loan, ready for: `EL = PD x LGD_final x EAD`

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.lgd_final import (
    build_final_lgd_layer,
    load_repo_portfolio_inputs,
    summarise_final_lgd_by_product,
    validate_final_lgd_layer,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

print("Setup complete.")

## Core Methodology Update
- Product summaries are EAD-weighted (`Sum(LGD*EAD)/Sum(EAD)`) for interview-ready portfolio interpretation.
- Discount-rate assumptions in upstream modules use `max(contract_rate_proxy, cost_of_funds_proxy)` by product.
- Final-layer downturn LGD is product-specific and macro-linked; it is not a single flat scalar across the portfolio.


In [ ]:
portfolio_inputs = load_repo_portfolio_inputs()
final_lgd = build_final_lgd_layer(portfolio_inputs)
summary = summarise_final_lgd_by_product(final_lgd)
checks = validate_final_lgd_layer(final_lgd)

print(f"Total loans in final LGD layer: {len(final_lgd)}")
print(f"Products: {final_lgd['source_product'].value_counts().to_dict()}")
final_lgd[["loan_id","source_product","product_type","security_type","industry",
           "lgd_base","lgd_adjusted","lgd_downturn","lgd_final"]].head(15)

In [ ]:
print("=== EAD-Weighted LGD by Product ===")
print(summary.to_string(index=False))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: EAD-weighted final LGD by source product
prod_lgd = (
    final_lgd.groupby("source_product", observed=True)
    .apply(lambda g: ((g["lgd_final"] * g["ead"]).sum() / g["ead"].sum()) if g["ead"].sum() else 0.0, include_groups=False)
    .sort_values()
)
colors = {"Mortgage": "#3498db", "Commercial": "#2ecc71", "Development": "#f39c12", "Cashflow Lending": "#e74c3c"}
bar_colors = [colors.get(p, "#95a5a6") for p in prod_lgd.index]
axes[0].barh(prod_lgd.index, prod_lgd.values, color=bar_colors)
axes[0].set_xlabel("EAD-Weighted Final LGD")
axes[0].set_title("EAD-Weighted LGD by Product")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
for i, (idx, val) in enumerate(prod_lgd.items()):
    axes[0].text(val + 0.005, i, f"{val:.1%}", va="center", fontsize=10)

# Histogram overlay
for prod in ["Mortgage", "Commercial", "Development", "Cashflow Lending"]:
    subset = final_lgd[final_lgd["source_product"] == prod]["lgd_final"]
    if len(subset) > 0:
        axes[1].hist(subset, bins=25, alpha=0.5, label=prod, color=colors.get(prod), density=True)
axes[1].set_xlabel("Final LGD")
axes[1].set_title("LGD Distribution by Product")
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
print("=== Validation Checks ===")
print(checks.to_string(index=False))

# Cash flow lending PD band breakdown
cfl = final_lgd[final_lgd["source_product"] == "Cashflow Lending"]
if len(cfl) > 0:
    print("
=== Cash Flow Lending: LGD by PD Score Band (EAD-Weighted) ===")
    rows = []
    for band, grp in cfl.groupby("pd_score_band", observed=True):
        rows.append({
            "pd_score_band": band,
            "count": len(grp),
            "total_ead": grp["ead"].sum(),
            "ead_weighted_lgd_base": ((grp["lgd_base"] * grp["ead"]).sum() / grp["ead"].sum()) if grp["ead"].sum() else 0.0,
            "ead_weighted_lgd_adjusted": ((grp["lgd_adjusted"] * grp["ead"]).sum() / grp["ead"].sum()) if grp["ead"].sum() else 0.0,
            "ead_weighted_lgd_final": ((grp["lgd_final"] * grp["ead"]).sum() / grp["ead"].sum()) if grp["ead"].sum() else 0.0,
        })
    band_summary = pd.DataFrame(rows).set_index("pd_score_band")
    print(band_summary.to_string())


In [ ]:
output_dir = os.path.join(os.getcwd(), '..', 'outputs', 'tables')
os.makedirs(output_dir, exist_ok=True)

final_lgd.to_csv(os.path.join(output_dir, 'lgd_final.csv'), index=False)
summary.to_csv(os.path.join(output_dir, 'lgd_final_summary_by_product.csv'), index=False)
checks.to_csv(os.path.join(output_dir, 'lgd_final_validation_checks.csv'), index=False)

portfolio_weighted_lgd = ((final_lgd['lgd_final'] * final_lgd['ead']).sum() / final_lgd['ead'].sum()) if final_lgd['ead'].sum() else 0.0

print(f"Saved {len(final_lgd)} rows to {output_dir}/lgd_final.csv")
print(f"
Final LGD layer summary:")
print(f"  Products:     {final_lgd['source_product'].nunique()}")
print(f"  Total loans:  {len(final_lgd)}")
print(f"  Total EAD:    ${final_lgd['ead'].sum():,.0f}")
print(f"  EAD-weighted LGD: {portfolio_weighted_lgd:.1%}")
print(f"
Ready for EL engine: EL = PD x LGD_final x EAD")
